<a href="https://colab.research.google.com/github/Jyotsna135-bit/llm-deployment-ollama/blob/main/ollama_deployment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LLM Deployment using Ollama
Running a local LLM on Colab using Ollama and querying it via the OpenAI-compatible API.

In [1]:
# zstd is needed by the ollama installer
!apt-get install -y zstd -qq
!curl -fsSL https://ollama.com/install.sh | sh

Selecting previously unselected package zstd.
(Reading database ... 122402 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [2]:
import subprocess, time, requests

# start ollama in the background
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# wait until it's ready
for _ in range(15):
    try:
        if requests.get("http://localhost:11434").status_code == 200:
            print("ollama is running")
            break
    except:
        time.sleep(1)

ollama is running


In [3]:
# pulling llama3.2:1b — small enough for the free T4 GPU (~1.3 GB)
!ollama pull llama3.2:1b

## curl client
Ollama exposes an OpenAI-compatible API, so the same `/v1/chat/completions` endpoint works.

In [4]:
%%bash
curl -s http://localhost:11434/v1/chat/completions \
  -H "Content-Type: application/json" \
  -H "Authorization: Bearer ollama" \
  -d '{
    "model": "llama3.2:1b",
    "messages": [
      {"role": "user", "content": "What is a large language model? Keep it short."}
    ]
  }' | python3 -c "import json,sys; print(json.load(sys.stdin)['choices'][0]['message']['content'])"

A large language model (LLM) is a software program that uses artificial intelligence to generate human-like text based on input or prompts. It can process and understand vast amounts of data, including words, phrases, and sentences, to produce coherent and relevant responses. LLMs are trained on massive datasets, such as books, articles, and conversations, allowing them to learn patterns and relationships in language.


## Python client
Using the `openai` SDK — just pointing `base_url` to the local Ollama server instead of OpenAI.

In [5]:
!pip install openai -q

In [6]:
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama",  # ollama doesn't need a real key
)

response = client.chat.completions.create(
    model="llama3.2:1b",
    messages=[
        {"role": "user", "content": "What is a large language model? Keep it short."}
    ]
)

print(response.choices[0].message.content)

A Large Language Model (LLM) is a computer program that uses deep learning algorithms to process and generate human-like natural language text, such as words, sentences, and even entire documents. It's an artificial intelligence system designed to analyze and understand the complexities of language while generating responses that are often informative, engaging, or entertaining.


In [7]:
# streaming response
stream = client.chat.completions.create(
    model="llama3.2:1b",
    messages=[
        {"role": "user", "content": "Explain how Ollama works in simple terms."}
    ],
    stream=True
)

for chunk in stream:
    text = chunk.choices[0].delta.content
    if text:
        print(text, end="", flush=True)

You're referring to Olam International, a global agribusiness group that helps farmers sell their products overseas.

In simple terms, Olam acts as an intermediary between buyers and sellers that operate in different countries or markets with incompatible currencies (e.g., USD for China). Here's how it works:

1. Olam connects buyers and sellers: It facilitates two-way trade between companies operating in multiple regions.
2. Exchange rates are matched: Olam tries to match the exchange rate of a US dollar per commodity (e.g., rice, coffee) with the wholesale market price of that commodity in other countries where they might be sold.
3. Wholesale market prices are locked in: The buyer in one country purchases these commodities at the agreed-upon price from an Olam-recommended supplier and later sells them at a higher price on their local currency markets (like the Chinese yuan or Indian rupee).
4. Olam earns commissions: For every unit of currency they facilitate, Olam earns a payment.


In [8]:
# multi-turn conversation
messages = [{"role": "system", "content": "You are a helpful assistant."}]

def chat(user_input):
    messages.append({"role": "user", "content": user_input})
    res = client.chat.completions.create(model="llama3.2:1b", messages=messages)
    reply = res.choices[0].message.content
    messages.append({"role": "assistant", "content": reply})
    return reply

print(chat("What is the transformer architecture?"))
print(chat("What role does attention play in it?"))

The Transformer Architecture is a revolutionary neural network architecture introduced in the paper "Attention Is All You Need" by Vaswani et al. in 2017. It's a natural language processing (NLP) model that has gained significant popularity due to its excellent performance in various NLP tasks.

The core idea behind the Transformer is to process sequences of tokens (words, characters, or symbols) simultaneously and jointly, unlike traditional recurrent neural networks (RNNs) and long short-term memory (LSTM) networks that rely on sequential dependencies. Instead, the Transformer uses self-attention mechanisms, which allow it to weigh the importance of each token based on its contextual similarity to other tokens.

The architecture consists of several key components:

1. **Input IDs**: A sequence of numbers representing the input sequences.
2. **Encoder Layers**: The first few layers are responsible for generating the representation ( Embeddings ) of the input input IDs. Each layer cons